# Phân vùng polyp: chạy toàn bộ thí nghiệm trên Colab

1. Tải thư mục `Project` lên Google Drive (ví dụ `MyDrive/advanced_ml/Project`).
2. Runtime → Change runtime type → **T4 GPU**.
3. Chạy lần lượt các ô. Mỗi lệnh dừng ô ngay khi lỗi, nên lỗi hiện ra là lỗi đầu tiên.
4. `outputs/` nằm trên Drive nên không mất khi bị ngắt phiên; chạy lại ô huấn luyện sẽ tự tiếp tục từ `last.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT  = '/content/drive/MyDrive/advanced_ml/Project'    # sửa nếu bạn để ở chỗ khác
ZIP_DIR  = '/content/drive/MyDrive/advanced_ml/data_zips'  # giữ file zip dữ liệu giữa các phiên
DATA_ROOT = '/content/data/polyp'                          # ổ cục bộ của Colab: đọc ảnh nhanh hơn Drive
%cd {PROJECT}

def run(cmd):
    """Chạy lệnh shell, in output, và dừng ô nếu lệnh lỗi."""
    print(f'$ {cmd}')
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get('_exit_code', 0)
    if code:
        raise RuntimeError(f'Lệnh lỗi (exit code {code}): {cmd}\nĐọc thông báo lỗi ngay phía trên.')

run('nvidia-smi --query-gpu=name,memory.total --format=csv')

In [ ]:
run('pip install -q timm==1.0.30 albumentations==2.0.8 gdown')
run('python -m pytest tests -q -p no:cacheprovider')

## Dữ liệu
Lần đầu, script tải hai file zip của PraNet (khoảng 730 MB) và lưu một bản vào `ZIP_DIR` trên Drive; các phiên sau chỉ giải nén từ Drive.

Nếu báo **không tải được** (link PraNet hay bị Google giới hạn lượt tải): mở link được in ra, tải zip bằng trình duyệt, đặt vào `MyDrive/advanced_ml/data_zips/` với tên `TrainDataset.zip` và `TestDataset.zip`, rồi chạy lại ô này.

In [ ]:
run(f'python scripts/download_data.py --dest {DATA_ROOT} --zip-dir {ZIP_DIR}')
run(f'python scripts/prepare_splits.py --root {DATA_ROOT}')
run(f'python scripts/check_duplicates.py --root {DATA_ROOT}')

## Huấn luyện baseline
Mô hình nào đã có kết quả sẽ được bỏ qua; mô hình đang dở sẽ tự resume từ `last.pt`. Mỗi run dừng sớm nếu Dice validation không tăng sau 20 epoch (tối đa 100 epoch). Mỗi phiên mới cần chạy lại ô đầu tiên và ô Dữ liệu.

In [ ]:
import os
MODELS = ['unet', 'multiresunet', 'cenet', 'transunet', 'swinunet']
for m in MODELS:
    if os.path.exists(f'outputs/metrics/per_image_{m}.csv'):
        print(f'== {m}: đã train và evaluate xong, bỏ qua'); continue
    run(f'python scripts/train.py --config configs/{m}.yaml --opts data.root={DATA_ROOT}')
    run(f'python scripts/evaluate.py --run outputs/runs/{m} --save-pred')


## Ablation trên baseline tốt nhất (RQ3)
Đặt `BEST` theo bảng kết quả baseline.

In [ ]:
BEST = 'cenet'
for a in ['c3_domain_aug', 'c2_boundary', 'c2_c3']:
    if os.path.exists(f'outputs/metrics/per_image_{BEST}_{a}.csv'):
        print(f'== {BEST}_{a}: đã xong, bỏ qua'); continue
    run(f'python scripts/train.py --config configs/{BEST}.yaml configs/ablation/{a}.yaml --opts data.root={DATA_ROOT}')
    run(f'python scripts/evaluate.py --run outputs/runs/{BEST}_{a}')


## Thống kê, chi phí tính toán và hình cho báo cáo

In [ ]:
REFERENCE = 'cenet'   # CNN tốt nhất, dùng cho cột p-value của Bảng 5
models = ' '.join(MODELS)
run(f'python scripts/stats_tests.py --models {models} --reference {REFERENCE} '
    f'--ablation {BEST} {BEST}_c3_domain_aug {BEST}_c2_boundary {BEST}_c2_c3')
run('python scripts/benchmark.py --configs ' + ' '.join(f'configs/{m}.yaml' for m in MODELS))
run(f'python scripts/plot_dataset_examples.py --root {DATA_ROOT}')
run(f'python scripts/plot_report_figures.py --models {models} --reference {REFERENCE} --root {DATA_ROOT}')
run('ls outputs/tables report/figures')

## Demo

In [ ]:
run('pip install -q gradio')
run(f'python app/gradio_app.py --run outputs/runs/{BEST} --share')